# Huấn luyện MobileNetV2 trên Google Colab

Sao chép mã nguồn dự án và dataset vào Google Drive trước. Notebook không tự tải, giải nén hay xoá dataset. Đặt project tại `MyDrive/waste_classification`, kiểm tra `data.dataset_dir` và `models.mobilenet.weights` trong cấu hình. ImageNet weights được tải khi cần. Có thể chọn GPU bằng **Runtime → Change runtime type → T4 GPU**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os

PROJECT_ROOT = Path('/content/drive/MyDrive/waste_classification')
if not (PROJECT_ROOT / 'configs' / 'config_mwcd.yaml').is_file():
    raise FileNotFoundError(f'Copy the project to Google Drive first: {PROJECT_ROOT}')
os.chdir(PROJECT_ROOT)
print('Project:', PROJECT_ROOT)

In [ ]:
%pip install -r requirements.txt

In [ ]:
import tensorflow as tf
from src.training.common import load_config, run_training
from src.data.dataset_loader import inspect_dataset

config = load_config('configs/config_mwcd.yaml')
print('TensorFlow:', tf.__version__)
print('Available GPUs:', tf.config.list_physical_devices('GPU'))
report = inspect_dataset(config['data']['dataset_dir'], config['data']['class_names'], config['data'].get('class_sources'))
print('Dataset:', report.dataset_root)
print('Images per class:', report.class_counts)
print('Invalid files:', len(report.invalid_files))
if not report.is_valid:
    raise ValueError('\n'.join(report.errors))

In [ ]:
# Head training first; change to True only when fine-tuning is desired.
model_path, metadata = run_training(
    'configs/config_mwcd.yaml',
    'mobilenet',
    fine_tune=False,
)
print('Saved model:', model_path)
print('Test metrics:', metadata['test_metrics'])